# Marketing Attribution — Analysis Notebook

Interactive exploration of the 5-model attribution comparison.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import snowflake.connector

load_dotenv(Path('..').resolve() / '.env')
sns.set_theme(style='whitegrid')

def conn():
    return snowflake.connector.connect(
        account=os.environ['SNOWFLAKE_ACCOUNT'],
        user=os.environ['SNOWFLAKE_USER'],
        password=os.environ['SNOWFLAKE_PASSWORD'],
        role=os.environ.get('SNOWFLAKE_ROLE','DBT_ROLE'),
        warehouse=os.environ.get('SNOWFLAKE_WAREHOUSE','DBT_WH_XS'),
        database=os.environ.get('SNOWFLAKE_DATABASE','MARKETING_DB'),
    )

def q(sql):
    c = conn(); cur = c.cursor(); cur.execute(sql)
    cols = [d[0].lower() for d in cur.description]
    df = pd.DataFrame(cur.fetchall(), columns=cols)
    cur.close(); c.close(); return df

## Same conversion, five attributions

In [ ]:
q('SELECT * FROM MARTS.MART_CHANNEL_ROAS_COMPARED ORDER BY model, channel')

## Spread between models per channel

In [ ]:
q('''
  SELECT channel,
         MIN(attributed_share) AS min_share,
         MAX(attributed_share) AS max_share,
         MAX(attributed_share) - MIN(attributed_share) AS spread
    FROM MARTS.MART_CHANNEL_ROAS_COMPARED
   GROUP BY 1 ORDER BY spread DESC
''')

## Bootstrap experiment results — Friedman test per channel

In [ ]:
m = q('SELECT * FROM META.experiment_attribution')
rows = []
for ch in m['channel'].unique():
    sub = m[m['channel']==ch].pivot_table(index='run_uuid', columns='model',
                                          values='attributed_share', aggfunc='first').dropna()
    if len(sub) < 5 or len(sub.columns) < 3: continue
    stat, p = stats.friedmanchisquare(*[sub[c] for c in sub.columns])
    rows.append({'channel': ch, 'friedman_stat': stat, 'p_value': p,
                 'significant': p < 0.05})
pd.DataFrame(rows)

## Budget reallocation recommendations

In [ ]:
q('SELECT model, channel, current_spend, recommended_spend, recommended_change_pct FROM MARTS.MART_BUDGET_REALLOCATION ORDER BY model, recommended_change_pct DESC')